# 模型矩阵 dashboard refresh 调度 v1

目标：验证 `src/schedule` 能把模型矩阵刷新和 dashboard export 串成一个可锁定、可通知、可重试的 job。

边界：本 notebook 使用轻量模型集合验证编排能力；生产级参数应放入 production notebook 或配置。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = Path('research/schedule_model_dashboard_refresh_v1')
CARD.mkdir(parents=True, exist_ok=True)
notifications = []
config = ModelDashboardPipelineConfig(
    name='model_dashboard_refresh_v1',
    card_dir=CARD / 'matrix_card',
    dashboard_output_path=CARD / 'dashboard_export.json',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'random_forest_clf'),
    top_n_factors=3,
    dashboard_top_n=5,
    retries=1,
)
config

ModelDashboardPipelineConfig(name='model_dashboard_refresh_v1', card_dir=PosixPath('research/schedule_model_dashboard_refresh_v1/matrix_card'), dashboard_output_path=PosixPath('research/schedule_model_dashboard_refresh_v1/dashboard_export.json'), start='2026-01-01', end='2026-06-09', train_end='2026-04-30', model_names=('ridge_mse', 'random_forest_clf'), top_n_factors=3, dashboard_top_n=5, retries=1)

In [2]:
result = run_model_dashboard_refresh_once(
    config,
    lock_dir=CARD / 'locks',
    notify=notifications.append,
)
summary = {
    'schema_version': 'schedule_model_dashboard_refresh.v1',
    'job_name': result.name,
    'status': result.status,
    'payload': result.payload,
    'notifications': notifications,
    'matrix_card': str(config.card_dir),
    'dashboard_export': str(config.dashboard_output_path),
    'lock_exists_after_run': (CARD / 'locks' / f'{config.name}.lock').exists(),
}
(CARD / 'run_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'schedule_model_dashboard_refresh.v1',
 'job_name': 'model_dashboard_refresh_v1',
 'status': 'success',
 'payload': {'matrix_card': 'research/schedule_model_dashboard_refresh_v1/matrix_card',
  'dashboard_export': 'research/schedule_model_dashboard_refresh_v1/dashboard_export.json',
  'dashboard_schema_version': 'dashboard.model_matrix.v1',
  'top_model': 'random_forest_clf__all_features',
  'matrix_status': 'feature_set_matrix_completed',
  'variant_count': 6,
  'attempts': 1},
 'notifications': ['model_dashboard_refresh_v1:success'],
 'matrix_card': 'research/schedule_model_dashboard_refresh_v1/matrix_card',
 'dashboard_export': 'research/schedule_model_dashboard_refresh_v1/dashboard_export.json',
 'lock_exists_after_run': False}

In [3]:
dashboard_payload = json.loads((CARD / 'dashboard_export.json').read_text(encoding='utf-8'))
{
    'schema_version': dashboard_payload['schema_version'],
    'top_models': dashboard_payload['top_models'][:3],
    'feature_set_summary': dashboard_payload['feature_set_summary'],
    'artifacts': dashboard_payload['artifacts'],
}

{'schema_version': 'dashboard.model_matrix.v1',
 'top_models': [{'rank': 1,
   'model': 'random_forest_clf__all_features',
   'base_model': 'random_forest_clf',
   'feature_set': 'all_features',
   'return_pct': 49.032787769965715,
   'benchmark_return_pct': 62.1472468532841,
   'relative_return_pct': -13.114459083318389,
   'max_drawdown_pct': 21.677243377022705,
   'sharpe': 3.8534654423822032,
   'avg_position': 0.7407635422717518,
   'turnover': 3.1283151044281827,
   'stability_score': 54.214257763652995},
  {'rank': 2,
   'model': 'ridge_mse__top_ic_factors',
   'base_model': 'ridge_mse',
   'feature_set': 'top_ic_factors',
   'return_pct': 51.68889425463847,
   'benchmark_return_pct': 62.1472468532841,
   'relative_return_pct': -10.458352598645632,
   'max_drawdown_pct': 26.17632325678787,
   'sharpe': 3.8153069361698657,
   'avg_position': 0.7496923542352576,
   'turnover': 2.1228598727860692,
   'stability_score': 52.875839799780145},
  {'rank': 3,
   'model': 'ridge_mse__ic_w

In [4]:
sorted(path.name for path in (CARD / 'matrix_card').iterdir())

['analysis_summary.json',
 'daily_backtest.csv',
 'factor_ic.csv',
 'feature_set_specs.json',
 'model_comparison.csv',
 'model_effect_deltas.csv',
 'predictions.csv',
 'qlib_risk_metrics.csv',
 'regime_metrics.csv',
 'window_metrics.csv']

## 中文结论

- 做了什么：通过 `src/schedule` 执行模型矩阵刷新，并自动生成 dashboard export。
- 核心指标：job status=`success`，attempts=1，通知=['model_dashboard_refresh_v1:success']。
- 输出：matrix card 位于 `research/schedule_model_dashboard_refresh_v1/matrix_card`；dashboard export 位于 `research/schedule_model_dashboard_refresh_v1/dashboard_export.json`。
- 锁边界：运行结束后 lock 是否存在：`False`。
- 是否晋升：否。本轮验证 schedule 编排能力，不代表模型方案生产晋升。
- 风险：真实生产调度还需要外部 cron/daemon 和失败告警通道；当前框架已提供单次 job、lock、retry、notify 入口。
- 下一步：补 `src/api` paper/sim 与 dashboard 中选出的 target position/策略意图衔接。